# Stage 5: saved primary shock comparisons

This notebook reads reviewed primary evaluation aggregates. **Run All is free:** it does not construct an API client or launch episodes. Counts refer to completed validated episodes; the saved status lists any missing matrix positions. The earlier readiness notebook is a historical Stage 4 checkpoint.

The primary window is 22 September–12 October 2023. Its prior manual outcome use is uncertain. There are three selected matched series-window blocks with dependent daily rows and one LLM realization per condition. These are synthetic simulator outcomes, not real-world causal estimates. Fallback days remain part of the operational LLM policy.

In [1]:
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'configs/study.json').exists():
    ROOT = ROOT.parent
PUBLIC = ROOT / 'docs/results/stage5'
status = json.loads((PUBLIC / 'status.json').read_text())
qc = json.loads((PUBLIC / 'qc_summary.json').read_text())
coverage = pd.read_csv(PUBLIC / 'coverage.csv')
daily = pd.read_csv(PUBLIC / 'daily_results.csv')
print(f"{status['status']}: {len(coverage)}/36 episodes, {len(daily)}/756 policy-days")
print('Missing matrix positions:', status['missing_matrix_indices'])


complete: 36/36 episodes, 756/756 policy-days
Missing matrix positions: []


In [2]:
verified = subprocess.run([sys.executable, 'scripts/check_public_stage5.py'], cwd=ROOT, check=True, capture_output=True, text=True)
print(verified.stdout.strip())
pd.DataFrame({'check': list(qc['checks']), 'passed': list(qc['checks'].values())})


Historical Stage 5 preparation passed: 36 frozen launch commands; original snapshot preserved.
Stage 5 saved execution evidence passed: 36/36 episodes, 756 policy-days; financial, reliability, matching and cost totals reconcile.


,check,passed
0,frozen_design_unchanged,True
1,execution_code_unchanged_since_preparation,True
2,all_complete_episodes_independently_replayed,True
3,matched_initial_conditions_and_base_paths,True
4,identical_base_paths_across_scenarios,True
5,exact_policy_requests_and_known_usage,True
6,all_attempts_and_held_charges_reconciled,True
7,fallback_and_failure_reviews_current,True
8,cash_inventory_pipeline_reconciled,True
9,no_duplicate_complete_episodes,True


## Complete episode evidence

Contribution is in assumed simulator monetary units; gateway expense is reported separately in credits. A lower or higher contribution is a descriptive matched outcome. Detailed control-adjusted effects and robustness analysis belong to Stage 6.

In [3]:
table = coverage[['store_id', 'product_id', 'scenario', 'policy', 'contribution', 'fill_rate', 'stockout_sku_days', 'mean_closing_inventory', 'repair_days', 'fallback_days']]
table.round(4)


,store_id,product_id,scenario,policy,contribution,fill_rate,stockout_sku_days,mean_closing_inventory,repair_days,fallback_days
0,S001,P0001,control,fixed,45691.1700,1.0000,0,202.6667,0,0
1,S001,P0001,control,adaptive,45677.5856,1.0000,0,219.0952,0,0
2,S001,P0001,control,llm_with_fallback,41922.1031,1.0000,0,1721.1905,8,2
3,S001,P0001,demand_surge,fixed,52916.0362,0.9619,1,163.5238,0,0
4,S001,P0001,demand_surge,adaptive,55106.3887,0.9868,1,228.8571,0,0
5,S001,P0001,demand_surge,llm_with_fallback,51304.5225,1.0000,0,1858.4762,10,2
6,S001,P0001,demand_decline,fixed,40101.1275,1.0000,0,232.9524,0,0
7,S001,P0001,demand_decline,adaptive,40111.9556,1.0000,0,219.8571,0,0
8,S001,P0001,demand_decline,llm_with_fallback,37296.0656,1.0000,0,1665.0000,7,1
9,S001,P0001,delivery_disruption,fixed,41294.7281,0.9033,3,154.5238,0,0


In [4]:
paired = coverage.pivot(index=['store_id', 'product_id', 'scenario'], columns='policy', values='contribution')
paired['llm_minus_adaptive'] = paired['llm_with_fallback'] - paired['adaptive']
paired.round(4)


policy                                     adaptive       fixed  \
store_id product_id scenario                                      
S001     P0001      control              45677.5856  45691.1700   
                    delivery_disruption  42575.0456  41294.7281   
                    demand_decline       40111.9556  40101.1275   
                    demand_surge         55106.3887  52916.0362   
S003     P0016      control              75769.0677  75418.4074   
                    delivery_disruption  75411.9482  71175.8495   
                    demand_decline       67985.5407  67601.5873   
                    demand_surge         89004.6287  88395.0989   
S004     P0006      control              31802.6739  31287.1632   
                    delivery_disruption  29609.2420  28836.3563   
                    demand_decline       28043.4284  27511.8826   
                    demand_surge         38789.2826  37060.2578   

policy                                   llm_with_fallback  llm_minus_adaptive  
store_id product_id scenario                                                    
S001     P0001      control                     41922.1031          -3755.4825  
                    delivery_disruption         43986.7050           1411.6594  
                    demand_decline              37296.0656          -2815.8900  
                    demand_surge                51304.5225          -3801.8662  
S003     P0016      control                     70592.6360          -5176.4317  
                    delivery_disruption         71035.5087          -4376.4394  
                    demand_decline              62983.4712          -5002.0695  
                    demand_surge                84309.3846          -4695.2442  
S004     P0006      control                     30152.8966          -1649.7773  
                    delivery_disruption         29870.7996            261.5576  
                    demand_decline              26639.1223          -1404.3062  
                    demand_surge                35774.5961          -3014.6864

## Reliability, costs and retained failures

First-pass validity, completion after the one allowed repair and adaptive fallback are distinct. Completed primary decision opportunities exclude partial failed cohorts; all physical attempts and charges include those cohorts. A review retains the frozen policy and poor outcomes; it does not delete fallback days or authorize new prompts.

In [5]:
reliability = {key: qc[key] for key in ['completed_llm_episodes', 'completed_primary_decision_opportunities', 'first_pass_valid_decisions', 'model_decisions_after_repair', 'repair_days', 'fallback_days', 'invalid_responses', 'successful_cohort_attempts', 'all_primary_attempts', 'retained_incomplete_cohorts']}
display(pd.Series(reliability, name='observed'))
display(pd.read_csv(PUBLIC / 'retained_failures.csv'))
display(pd.Series(status['budget_snapshot'], name='cumulative_accounting'))
print(status['interpretation'])
print(status['native_coverage_qualification'])


completed_llm_episodes                       12
completed_primary_decision_opportunities    252
first_pass_valid_decisions                  108
model_decisions_after_repair                235
repair_days                                 144
fallback_days                                17
invalid_responses                           161
successful_cohort_attempts                  396
all_primary_attempts                        396
retained_incomplete_cohorts                   0
Name: observed, dtype: int64

,matrix_index,status,failure_category,started_attempts,known_cost_credits,fully_held_unknown_credits,evidence_sha256,review_decision


currency                                                                 USD_gateway_credit
inclusive_cap_credits                                                                  10.0
native_scope_cap_credits                                                                4.2
kaggle_scope_cap_credits                                                                3.8
unallocated_reserve_credits                                                             2.0
total_project_attempts                                                                 1834
inclusive_project_exposure_credits                                                 2.186207
known_project_cost_credits                                                         2.127123
retained_unknown_attempts                                                                 8
retained_unknown_charge_credits                                                    0.059084
all_primary_attempts                                                            

Three matched series-window blocks with dependent daily rows; one LLM realization per condition. Synthetic simulator outcomes, not real-world causal evidence.
14 validated non-Retail tasks; ICBE validity failure retained; four Retail tasks deferred.
